# PINN European Call Experiment

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / "Thesis" / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT / "Thesis"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.black_scholes import black_scholes_call
from src.finite_differences import crank_nicolson_call
from src.monte_carlo import monte_carlo_european_call
from src.pinn import LossWeights, PINN, compare_t0_curve, plot_loss_history, plot_t0_comparison, price_pinn, train_pinn


In [ ]:
K = 100.0
T = 1.0
r = 0.03
sigma = 0.2
S_max = 300.0
S0 = 100.0
t0 = 0.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PINN().to(device)


In [ ]:
model, training_metrics = train_pinn(
    model,
    K=K,
    T=T,
    r=r,
    sigma=sigma,
    S_max=S_max,
    epochs=5000,
    n_interior=2000,
    n_terminal=800,
    n_boundary=400,
    lr=1e-3,
    device=device,
    print_every=500,
)


In [ ]:
plot_loss_history(training_metrics)
plt.show()


In [ ]:
price_pinn_value = price_pinn(model, S0, t0, S_max, T, device=device)
price_bs = black_scholes_call(S0, K, T, r, sigma)
price_mc = monte_carlo_european_call(S0, K, T, r, sigma, n_paths=10000)
price_cn, _, _, _ = crank_nicolson_call(S0, K, T, r, sigma, S_max, M=200, N=200)

print("PINN price:", price_pinn_value)
print("Black-Scholes price:", price_bs)
print("Monte Carlo price:", price_mc)
print("Crank-Nicolson price:", price_cn)


## PINN Reliability Across Selected Volatilities

Train one fixed-volatility PINN for each selected sigma and compare the t=0 curve against the analytical Black-Scholes benchmark.

In [ ]:
selected_pinn_sigmas = [0.10, 0.25, 0.40]
pinn_seed = 123
pinn_reliability_epochs = 5000
pinn_training_S_max = 700.0
pinn_curve_S_max = 1.08 * pinn_training_S_max
S_curve = np.linspace(1e-8, pinn_curve_S_max, 101)
pinn_loss_weights = LossWeights(terminal=5.0, boundary_right=5.0)
pinn_value_scale = K

reliability_rows = []
reliability_models = {}
reliability_comparisons = {}

for sigma_i in selected_pinn_sigmas:
    model_i = PINN().to(device)
    model_i, metrics_i = train_pinn(
        model_i,
        K=K,
        T=T,
        r=r,
        sigma=sigma_i,
        S_max=pinn_training_S_max,
        epochs=pinn_reliability_epochs,
        n_interior=2000,
        n_terminal=800,
        n_boundary=400,
        lr=1e-3,
        device=device,
        print_every=0,
        seed=pinn_seed,
        loss_weights=pinn_loss_weights,
        value_scale=pinn_value_scale,
    )

    comparison_i = compare_t0_curve(model_i, S_curve, K, T, r, sigma_i, pinn_training_S_max, device=device, value_scale=pinn_value_scale)
    pinn_spot = price_pinn(model_i, S0, t0, pinn_training_S_max, T, device=device, value_scale=pinn_value_scale)
    analytical_spot = black_scholes_call(S0, K, T, r, sigma_i)
    spot_abs_error = abs(pinn_spot - analytical_spot)
    spot_rel_error = spot_abs_error / abs(analytical_spot)

    reliability_models[sigma_i] = model_i
    reliability_comparisons[sigma_i] = comparison_i
    reliability_rows.append(
        {
            "sigma": sigma_i,
            "pinn_price_at_S0": pinn_spot,
            "analytical_price_at_S0": analytical_spot,
            "spot_abs_error": spot_abs_error,
            "spot_rel_error": spot_rel_error,
            "curve_mae": comparison_i["mae"],
            "curve_rmse": comparison_i["rmse"],
            "curve_max_abs_error": comparison_i["max_abs_error"],
            "training_S_max": pinn_training_S_max,
            "curve_S_max": pinn_curve_S_max,
            "value_scale": pinn_value_scale,
            "runtime_sec": metrics_i["runtime_sec"],
            "final_loss": metrics_i["final_total_loss"],
        }
    )


In [ ]:
header = (
    "sigma | PINN@S0 | BS@S0 | abs err | rel err | curve MAE | curve RMSE | "
    "curve max err | runtime sec"
)
print(header)
print("-" * len(header))
for row in reliability_rows:
    print(
        f"{row['sigma']:.2f} | "
        f"{row['pinn_price_at_S0']:.6f} | "
        f"{row['analytical_price_at_S0']:.6f} | "
        f"{row['spot_abs_error']:.6f} | "
        f"{row['spot_rel_error']:.4%} | "
        f"{row['curve_mae']:.6f} | "
        f"{row['curve_rmse']:.6f} | "
        f"{row['curve_max_abs_error']:.6f} | "
        f"{row['runtime_sec']:.2f}"
    )


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, sigma_i in zip(axes, selected_pinn_sigmas):
    plot_t0_comparison(reliability_comparisons[sigma_i], ax=ax)
    ax.axvline(pinn_training_S_max, color="gray", linestyle=":", label="training boundary")
    ax.axvline(S0, color="navy", linestyle="--", linewidth=1.2, label=f"S₀={S0:.0f} (evaluation point)")
    ax.set_title(f"t=0 curve, sigma={sigma_i:.2f}")
    ax.legend()
plt.tight_layout()
plt.show()

### Note on the σ = 0.10 Result

The t=0 curve for σ = 0.10 *appears* to track Black-Scholes well across the full domain, yet the quantitative error at S₀ = 100 is approximately 111% (see the summary table above). The visual impression is misleading for a structural reason: at low volatility the option price near S₀ = 100 is only ~4.52 (the option is barely in-the-money with K = 110, T = 2, σ = 0.10), while prices grow steeply for high stock values. Because the y-axis is scaled to accommodate the full range S ∈ [0, 756], a discrepancy of ~5 dollars at S₀ = 100 is nearly invisible even though it represents more than a 100% relative error.

The root cause is a gradient-sharpness problem: at low volatility the Black-Scholes pricing function has a pronounced "kink" near the at-the-money region — transitioning sharply from near-zero to a positive intrinsic value — concentrated in a small sub-domain. The neural network, trained with uniformly sampled collocation points, sees relatively few points in this critical region and smooths over the curvature. This is a known limitation of PINNs for problems with steep localised gradients. At higher volatilities (σ = 0.25, 0.40) the payoff surface is smoother and the network converges to an accurate solution.

### Positioning PINNs in the Comparison

PINNs represent a promising research direction for option pricing, but the results above make clear they are not yet competitive with classical methods for vanilla European calls:

- **Black-Scholes** provides an exact analytical solution in microseconds.
- **Monte Carlo** scales naturally to path-dependent and high-dimensional problems (Asian options, basket options) at the cost of statistical noise.
- **Crank-Nicolson** solves the Black-Scholes PDE on a grid with O(h²) convergence in tens of milliseconds; it is the most accurate numerical method for 1D European options.
- **PINNs** require 50–65 seconds of training *per volatility*, and even then achieve errors 10–100× larger than Crank-Nicolson at comparable computational cost (see the error-vs-runtime frontier in the benchmark notebook).

The key theoretical advantage of PINNs — that a single *volatility-conditioned* network could price across all parameter combinations simultaneously — is not exploited here. Each PINN is trained for a single fixed σ, which eliminates this benefit. A natural extension would be to add σ as an explicit input to the network alongside S and t, enabling one training run to generalise across the full parameter space. Whether such a model can match Crank-Nicolson accuracy remains an open question and a direction for future work.

In [ ]:
sigmas_plot = [row["sigma"] for row in reliability_rows]
mae_values = [row["curve_mae"] for row in reliability_rows]
rmse_values = [row["curve_rmse"] for row in reliability_rows]
max_values = [row["curve_max_abs_error"] for row in reliability_rows]
runtime_values = [row["runtime_sec"] for row in reliability_rows]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sigmas_plot, mae_values, marker="o", label="MAE")
axes[0].plot(sigmas_plot, rmse_values, marker="o", label="RMSE")
axes[0].plot(sigmas_plot, max_values, marker="o", label="Max abs error")
axes[0].set_xlabel("Volatility")
axes[0].set_ylabel("Error vs Black-Scholes")
axes[0].set_title("PINN t=0 Curve Error")
axes[0].grid(True, linestyle=":", alpha=0.6)
axes[0].legend()

axes[1].bar([f"{sigma_i:.2f}" for sigma_i in sigmas_plot], runtime_values)
axes[1].set_xlabel("Volatility")
axes[1].set_ylabel("Runtime (seconds)")
axes[1].set_title("PINN Training Runtime")
axes[1].grid(True, axis="y", linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()
